# Feature pipeline and model smoke test

This notebook mounts Google Drive in Colab, reads the EB-NeRD Parquet files from `/content/drive/MyDrive/Colab Notebooks/ebnerd_large`, builds a small train and validation sample, and fits a simple logistic click model. It writes **one encoded Parquet file per split** and a small JSON file containing the one-hot mapping fitted on train.

In Colab, put `feature_pipeline.py` in `/content/drive/MyDrive/Colab Notebooks/`. Install `pyarrow`, `pandas`, `numpy`, and `scikit-learn` in the notebook kernel. The pipeline reads the large source Parquet files in batches; pandas loads only the small encoded smoke-test outputs.


In [17]:
from pathlib import Path
import sys

# Colab: access the same Drive folder used in the example paths below.
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, ndcg_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

# In Colab, load the script you uploaded beside the ebnerd_large folder.
# Outside Colab, find the script in the local project checkout.
PROJECT_ROOT = next(
    (root for root in [Path.cwd(), *Path.cwd().parents]
     if (root / 'Feature Engineering/feature_pipeline.py').is_file()),
    None,
)
if IN_COLAB:
    pipeline_path = Path('/content/drive/MyDrive/Colab Notebooks/feature_pipeline.py')
else:
    pipeline_path = (
        PROJECT_ROOT / 'Feature Engineering/feature_pipeline.py'
        if PROJECT_ROOT is not None else None
    )
if pipeline_path is None or not pipeline_path.is_file():
    raise FileNotFoundError(
        f'Cannot find feature_pipeline.py at {pipeline_path}. '
        'Check the file name and its Drive location.'
    )
sys.path.insert(0, str(pipeline_path.parent))
sys.modules.pop('feature_pipeline', None)
from feature_pipeline import (
    build_model_feature_file,
    encoded_baseline_columns,
    load_onehot_state,
)

# Source Parquet files on Drive; use the local data folder outside Colab.
DATA_ROOT = (
    Path('/content/drive/MyDrive/Colab Notebooks/ebnerd_large')
    if IN_COLAB else PROJECT_ROOT / 'data/ebnerd_large'
)
articles_path = DATA_ROOT / 'articles.parquet'
train_behaviors_path = DATA_ROOT / 'train/behaviors.parquet'
train_history_path = DATA_ROOT / 'train/history.parquet'
validation_behaviors_path = DATA_ROOT / 'validation/behaviors.parquet'
validation_history_path = DATA_ROOT / 'validation/history.parquet'
for source_path in (
    articles_path, train_behaviors_path, train_history_path,
    validation_behaviors_path, validation_history_path,
):
    if not source_path.is_file():
        raise FileNotFoundError(source_path)

OUT_DIR = (
    DATA_ROOT.parent / 'feature_smoke'
    if IN_COLAB else PROJECT_ROOT / 'data/feature_smoke'
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
USER_MODULUS = 100        # stable 1% user partition
USER_REMAINDER = 0
MAX_IMPRESSIONS = 1_000   # per split; small functionality sample
print('Source data:', DATA_ROOT)
print('Final outputs:', OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Source data: /content/drive/MyDrive/Colab Notebooks/ebnerd_large
Final outputs: /content/drive/MyDrive/Colab Notebooks/feature_smoke


In [18]:
# Build one final encoded file per split. Raw features are temporary and
# deleted after encoding. The train split fits the shared one-hot mapping.
state_path = OUT_DIR / 'onehot_state.json'
train_encoded_path = OUT_DIR / 'train_encoded.parquet'
validation_encoded_path = OUT_DIR / 'validation_encoded.parquet'

train_counts = build_model_feature_file(
    train_behaviors_path,
    train_history_path,
    articles_path,
    train_encoded_path,
    split='train',
    onehot_state_path=state_path,
    fit_onehot=True,
    label_mode='required',
    batch_size=500,
    user_modulus=USER_MODULUS,
    user_remainder=USER_REMAINDER,
    max_impressions=MAX_IMPRESSIONS,
    overwrite=True,
)
validation_counts = build_model_feature_file(
    validation_behaviors_path,
    validation_history_path,
    articles_path,
    validation_encoded_path,
    split='validation',
    onehot_state_path=state_path,
    label_mode='required',
    batch_size=500,
    user_modulus=USER_MODULUS,
    user_remainder=USER_REMAINDER,
    max_impressions=MAX_IMPRESSIONS,
    overwrite=True,
)
display(pd.DataFrame([
    {'split': 'train', **train_counts},
    {'split': 'validation', **validation_counts},
]))


,split,impressions,candidates,users_with_history,onehot_training_rows,rows,onehot_columns
0,train,1000,11029,7810,11029.0,11029,35
1,validation,1000,11495,7853,NaN,11495,35


In [19]:
# Load only the small final files for the model functionality check.
state = load_onehot_state(state_path)
train = pd.read_parquet(train_encoded_path)
validation = pd.read_parquet(validation_encoded_path)

assert train.columns.tolist() == validation.columns.tolist()
assert len(train) == train_counts['candidates']
assert len(validation) == validation_counts['candidates']
assert train['clicked'].notna().all() and validation['clicked'].notna().all()
assert not {'device_type', 'category', 'article_type', 'premium_subscriber'} & set(train.columns)

# The file retains all numeric/boolean features, including engineered ones.
# The example logistic model below selects a suggested subset from this file.
non_predictor_columns = {
    'split', 'impression_id', 'user_id', 'candidate_article_id',
    'current_page_article_id', 'impression_time', 'clicked',
}
available_predictors = [name for name in train if name not in non_predictor_columns]
default_baseline_predictors = list(encoded_baseline_columns(state))
additional_predictors = [
    name for name in available_predictors if name not in default_baseline_predictors
]
assert set(default_baseline_predictors) <= set(available_predictors)
assert {
    'slate_relative_age_h', 'category_affinity_72h',
    'last5_mean_topic_jaccard', 'history_clicks_7d',
    'same_category_mean_log1p_read_time',
} <= set(available_predictors)
assert any(name.startswith('ohe_premium_subscriber_') for name in available_predictors)

print('Encoded train shape:', train.shape)
print('Available predictors in file:', len(available_predictors))
print('Default baseline predictors:', len(default_baseline_predictors))
print('Additional predictors beyond default baseline:', additional_predictors)
print('Encoded validation shape:', validation.shape)
print('One-hot columns:', sum(name.startswith('ohe_') for name in train))
print('Final files:', train_encoded_path, validation_encoded_path, state_path, sep='\n')
display(train[[
    'impression_id', 'candidate_article_id', 'clicked',
    *[name for name in train if name.startswith('ohe_category_')][:4],
]].head())


Encoded train shape: (11029, 69)
Available predictors in file: 62
Default baseline predictors: 42
Additional predictors beyond default baseline: ['history_length_log1p', 'median_read_time_log1p', 'history_clicks_7d', 'history_clicks_7d_log1p', 'article_age_log1p', 'is_24h_eligible', 'slate_relative_age_h', 'category_affinity_72h', 'category_affinity_missing', 'last5_mean_topic_jaccard', 'topic_affinity_missing', 'has_any_topic_overlap', 'same_category_mean_log1p_read_time', 'same_category_read_depth_missing', 'ohe_premium_subscriber_000', 'ohe_premium_subscriber_001', 'ohe_premium_subscriber_002', 'ohe_premium_subscriber_003', 'ohe_premium_subscriber_missing', 'ohe_premium_subscriber_unknown']
Encoded validation shape: (11495, 69)
One-hot columns: 35
Final files:
/content/drive/MyDrive/Colab Notebooks/feature_smoke/train_encoded.parquet
/content/drive/MyDrive/Colab Notebooks/feature_smoke/validation_encoded.parquet
/content/drive/MyDrive/Colab Notebooks/feature_smoke/onehot_state.json


,impression_id,candidate_article_id,clicked,ohe_category_000,ohe_category_001,ohe_category_002,ohe_category_003
0,47829,9774557,0,0,0,1,0
1,47829,9775419,0,1,0,0,0
2,47829,9774516,0,0,0,0,0
3,47829,9775371,0,1,0,0,0
4,47829,9775331,0,1,0,0,0


In [22]:
# Stage 3: choose predictors, impute from train, then scale numeric values.
# Default = all EDA-retained original predictors, using raw numeric values.
# Toggle these to add engineered predictors or select log1p numeric variants.
include_engineered = True
log_transform = False
feature_columns = list(encoded_baseline_columns(
    state,
    include_engineered=include_engineered,
    log_transform=log_transform,
))
print('include_engineered:', include_engineered)
print('log_transform:', log_transform)
print('Selected model predictors:', feature_columns)
X_train = train[feature_columns].apply(pd.to_numeric, errors="coerce").astype("float64")
X_validation = validation[feature_columns].apply(pd.to_numeric, errors="coerce").astype("float64")
y_train = train["clicked"].astype(int)
y_validation = validation["clicked"].astype(int)

assert y_train.nunique() == 2, "The smoke sample needs both click classes."
assert y_validation.nunique() == 2, "The validation sample needs both click classes."

train_medians = X_train.median().fillna(0.0)
X_train = X_train.fillna(train_medians)
X_validation = X_validation.fillna(train_medians)

binary_columns = {
    "is_sso_user", "is_subscriber", "is_front_page", "has_history",
    "premium", "article_age_missing", "is_24h_eligible",
    "category_affinity_missing", "topic_affinity_missing",
    "same_category_read_depth_missing", "has_any_topic_overlap",
}
scale_columns = [
    name for name in feature_columns
    if name not in binary_columns and not name.startswith("ohe_")
]
scaler = StandardScaler()
X_train.loc[:, scale_columns] = scaler.fit_transform(X_train[scale_columns])
X_validation.loc[:, scale_columns] = scaler.transform(X_validation[scale_columns])

assert X_train.notna().all().all() and X_validation.notna().all().all()
print("Model matrix shapes:", X_train.shape, X_validation.shape)
print("Scaled numeric columns:", scale_columns)


include_engineered: True
log_transform: False
Selected model predictors: ['is_sso_user', 'is_subscriber', 'is_front_page', 'hour', 'day_of_week', 'slate_size', 'has_history', 'history_length', 'median_read_time', 'mean_scroll_percentage', 'premium', 'article_age_h', 'article_age_missing', 'history_clicks_7d', 'is_24h_eligible', 'slate_relative_age_h', 'category_affinity_72h', 'category_affinity_missing', 'last5_mean_topic_jaccard', 'topic_affinity_missing', 'has_any_topic_overlap', 'same_category_mean_log1p_read_time', 'same_category_read_depth_missing', 'ohe_device_type_000', 'ohe_device_type_001', 'ohe_device_type_002', 'ohe_device_type_missing', 'ohe_device_type_unknown', 'ohe_category_000', 'ohe_category_001', 'ohe_category_002', 'ohe_category_003', 'ohe_category_004', 'ohe_category_005', 'ohe_category_006', 'ohe_category_007', 'ohe_category_008', 'ohe_category_009', 'ohe_category_010', 'ohe_category_011', 'ohe_category_missing', 'ohe_category_unknown', 'ohe_article_type_000', 'ohe

In [23]:
# Stage 4: fit a small model and confirm that it produces scores and rankings.
model = LogisticRegression(max_iter=500, solver="liblinear")
model.fit(X_train, y_train)
validation_scores = model.predict_proba(X_validation)[:, 1]

ranked = validation[
    ["impression_id", "candidate_article_id", "clicked"]
].copy()
ranked["predicted_click_probability"] = validation_scores

ndcg_values = []
for _, slate in ranked.groupby("impression_id", sort=False):
    if len(slate) > 1:
        ndcg_values.append(
            ndcg_score(
                slate["clicked"].to_numpy()[None, :],
                slate["predicted_click_probability"].to_numpy()[None, :],
                k=min(10, len(slate)),
            )
        )
metrics = pd.DataFrame([{
    "validation_candidates": len(validation),
    "roc_auc": roc_auc_score(y_validation, validation_scores),
    "log_loss": log_loss(y_validation, validation_scores),
    "mean_ndcg_at_10": float(np.mean(ndcg_values)),
}])
display(metrics)

ranked = ranked.sort_values(
    ["impression_id", "predicted_click_probability"],
    ascending=[True, False],
)
ranked["predicted_rank"] = ranked.groupby("impression_id").cumcount() + 1
display(ranked.loc[ranked["predicted_rank"] <= 3].head(15))


,validation_candidates,roc_auc,log_loss,mean_ndcg_at_10
0,11495,0.741609,0.2679,0.56554


,impression_id,candidate_article_id,clicked,predicted_click_probability,predicted_rank
14,141377,9788203,0,0.191156,1
11,141377,9782005,0,0.180129,2
15,141377,9788045,1,0.107171,3
24,141378,9787176,0,0.107446,1
27,141378,9786557,0,0.089260,2
39,141378,9787593,0,0.087380,3
53,141386,9787499,0,0.300866,1
47,141386,9788557,0,0.252571,2
48,141386,9783042,0,0.173403,3
59,141387,9788125,0,0.369967,1


## What this checks

- The builder reads the Drive Parquet files in batches and writes one row per shown candidate. It holds the article catalogue and selected users’ history profiles in memory.
- Each split produces one final encoded Parquet file containing the features currently built by this pipeline. A temporary raw table is removed after encoding.
- The logistic example defaults to all EDA-retained original predictors. In the preprocessing cell, `include_engineered=True` adds engineered predictors and `log_transform=True` selects available log1p variants. These switches change model inputs, not the saved Parquet files.
- `onehot_state.json` stores the train-fitted category mapping so validation and future test data have the same columns.
- Imputation values and scaling parameters are fitted on train and then applied to validation, producing numeric matrices for the logistic model.
- The model scores and ranks all validation candidates within each impression; the displayed preview shows at most 15 rows and only ranks 1–3.

The notebook takes the first 1,000 impressions, in file order, from users with `user_id % 100 == 0` in each split. This deterministic subset is for a functionality test; its metrics are not a final performance estimate or a basis for feature selection.
